## Performance on describing final layer neurons of ResNet-50 (ImageNet)

In [1]:
import os
#virtually move to parent directory
os.chdir("..")

import torch
import pandas as pd
from sentence_transformers import SentenceTransformer

import clip
import utils
import similarity

/home/s4yadav/private/workspace/CLIP-dissect/clip/clip.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import packaging


## Arguments for CLIP-Dissect

In [2]:
clip_name = 'ViT-B/16'
target_name = 'resnet50'
target_layer = 'fc'
batch_size = 4
device = 'cuda'
pool_mode = 'avg'

save_dir = 'saved_activations'
similarity_fn = similarity.soft_wpmi

In [3]:
model = SentenceTransformer('all-mpnet-base-v2')
clip_model, _ = clip.load(clip_name, device=device)

with open('data/imagenet_labels.txt', 'r') as f: 
    imagenet_classnames = (f.read()).split('\n')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

/home/s4yadav/.conda/envs/clip-rtx/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## Run CLIP-Dissect

In [4]:
rows = [("imagenet_val", "data/broden_labels_clean.txt"),
       ("imagenet_val", "data/3k.txt"),
       ("imagenet_val", "data/10k.txt"),
       ("imagenet_val", "data/20k.txt"),
       ("imagenet_val", "data/imagenet_labels.txt"),
       ("cifar100_train", "data/20k.txt"),
       ("broden", "data/20k.txt"),
       ("imagenet_val", "data/20k.txt"),
       ("imagenet_broden", "data/20k.txt"),]

In [5]:
for d_probe, concept_set in rows:
    with open(concept_set, 'r') as f: 
        words = (f.read()).split('\n')
    utils.save_activations(clip_name = clip_name, target_name = target_name, target_layers = [target_layer], 
                           d_probe = d_probe, concept_set = concept_set, batch_size = batch_size, 
                           device = device, pool_mode=pool_mode, save_dir = save_dir)

    save_names = utils.get_save_names(clip_name = clip_name, target_name = target_name,
                                      target_layer = target_layer, d_probe = d_probe,
                                      concept_set = concept_set, pool_mode=pool_mode,
                                      save_dir = save_dir)

    target_save_name, clip_save_name, text_save_name = save_names

    similarities, target_feats = utils.get_similarity_from_activations(target_save_name, clip_save_name, 
                                                        text_save_name, similarity_fn, device=device)

    clip_preds = torch.argmax(similarities, dim=1)
    clip_preds = [words[int(pred)] for pred in clip_preds]

    clip_cos, mpnet_cos = utils.get_cos_similarity(clip_preds, imagenet_classnames, clip_model, model, device, batch_size)
    print("D_probe:{}, Concept set:{}".format(d_probe, concept_set))
    print("CLIP-Dissect - Clip similarity: {:.4f}, mpnet similarity: {:.4f}".format(clip_cos, mpnet_cos))

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /tmp/xdg-cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 180MB/s]
/home/s4yadav/private/workspace/CLIP-dissect/utils.py:153: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have 

torch.Size([1000, 1197])
D_probe:imagenet_val, Concept set:data/broden_labels_clean.txt
CLIP-Dissect - Clip similarity: 0.7393, mpnet similarity: 0.4198


100%|██████████| 1000/1000 [00:02<00:00, 436.19it/s]


torch.Size([1000, 3000])
D_probe:imagenet_val, Concept set:data/3k.txt
CLIP-Dissect - Clip similarity: 0.7456, mpnet similarity: 0.4166


100%|██████████| 1000/1000 [00:05<00:00, 169.69it/s]


...

torch.Size([1000, 20000])
D_probe:broden, Concept set:data/20k.txt
CLIP-Dissect - Clip similarity: 0.7417, mpnet similarity: 0.3966


100%|██████████| 1000/1000 [00:07<00:00, 141.23it/s]


torch.Size([1000, 20000])
D_probe:imagenet_val, Concept set:data/20k.txt
CLIP-Dissect - Clip similarity: 0.7900, mpnet similarity: 0.5263


100%|██████████| 1000/1000 [00:09<00:00, 105.16it/s]


torch.Size([1000, 20000])
D_probe:imagenet_broden, Concept set:data/20k.txt
CLIP-Dissect - Clip similarity: 0.7900, mpnet similarity: 0.5241


## Baselines

In [6]:
netdissect_res = pd.read_csv('data/NetDissect_results/resnet50_imagenet_fc.csv')
nd_preds = netdissect_res['label'].values

clip_cos, mpnet_cos = utils.get_cos_similarity(nd_preds, imagenet_classnames, clip_model, model, device, batch_size)
print("Network Dissection - Clip similarity: {:.4f}, mpnet similarity: {:.4f}".format(clip_cos, mpnet_cos))

Network Dissection - Clip similarity: 0.6929, mpnet similarity: 0.2952


In [7]:
milan_preds = pd.read_csv('data/MILAN_results/m_base_resnet50_imagenet.csv')
milan_preds = milan_preds[milan_preds['layer']=='fc']
milan_preds = milan_preds.sort_values(by=['unit'])
milan_preds = list(milan_preds['description'])

clip_cos, mpnet_cos = utils.get_cos_similarity(milan_preds, imagenet_classnames, clip_model, model, device, batch_size)
print("MILAN - Clip similarity: {:.4f}, mpnet similarity: {:.4f}".format(clip_cos, mpnet_cos))

MILAN - Clip similarity: 0.7080, mpnet similarity: 0.2788
